In [1]:
import os
import re
import json
import pickle
import random
import string

from copy import deepcopy
from collections import defaultdict, Counter
from transformers import AutoTokenizer

/home/anaconda3/envs/search_o1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def pickle_load(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

def pickle_dump(path, data):
    with open(path, "wb") as f:
        pickle.dump(data, f)

def json_load(path):
    with open(path, mode='r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def json_dump(path, data):
    with open(path, mode='w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [6]:
# hotpotqa distractor setting
dist_data = json_load("data/hotpotqa/hotpot_dev_distractor_v1.json")
for k,v in dist_data[0].items():
    print(k, v)
print(len(dist_data))

# shuffle
random.seed(42)
random.shuffle(dist_data)
dist_data = dist_data[:500]  # Limit to 500 samples for testing

dataset_name = 'hotpotqa'
os.makedirs(f'../QA_Datasets/{dataset_name}/', exist_ok=True)
output_path = f'../QA_Datasets/{dataset_name}/{dataset_name}_distractor_500.json'

doc_idx = 0
data_list = []
cache_list = defaultdict(list)
for item in dist_data:
    question = item['question']
    answer = [item['answer']]
    data_list.append({
        "id": item["_id"],
        "Question": question,
        "answer": answer,
        "type": item['type'],
        "level": item['level'],
    })
    doc_list = item['context']
    for doc in doc_list:
        cache_list[question].append({
            "id": doc_idx,
            "title": doc[0],
            "contents": " ".join(doc[1]),
        })
        doc_idx += 1
    
# Write the updated data to JSON
with open(output_path, mode='w', encoding='utf-8') as json_file:
    json.dump(data_list, json_file, indent=4, ensure_ascii=False)

os.makedirs(f'../../cache/{dataset_name}', exist_ok=True)
cache_path = f'../../cache/{dataset_name}/search_cache_distractor_500.json'
with open(cache_path, mode='w', encoding='utf-8') as json_file:
    json.dump(cache_list, json_file, indent=4, ensure_ascii=False)

_id 5a8b57f25542995d1e6f1371
answer yes
question Were Scott Derrickson and Ed Wood of the same nationality?
supporting_facts [['Scott Derrickson', 0], ['Ed Wood', 0]]
context [['Ed Wood (film)', ['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.', " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.", ' Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.']], ['Scott Derrickson', ['Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.', ' He lives in Los Angeles, California.', ' He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."']], ['Wood